# API Demo: Fetching Weather Data with Open-Meteo
**Computational Data Analysis with Python — Week 3**

In this notebook, you will:
1. Understand what a REST API request looks like
2. Run a minimal working example
3. Extend it yourself (two short exercises)

No API key is required. Open-Meteo is a free, open-source weather API.

---
## Part 1: What does an API request look like?

Before writing any code, open this URL in your browser:

```
https://api.open-meteo.com/v1/forecast?latitude=48.14&longitude=11.58&hourly=temperature_2m
```

You should see raw JSON. Notice the structure:
- `latitude`, `longitude` — location
- `hourly` — a dictionary with two keys: `time` and `temperature_2m`

Every key in `hourly` is a list of the same length. That is a DataFrame waiting to happen.

**Anatomy of the URL**

| Part | Meaning |
|---|---|
| `https://api.open-meteo.com/v1/forecast` | Endpoint — the address of the resource |
| `?` | Separator between endpoint and parameters |
| `latitude=48.14&longitude=11.58` | Query parameters — your specification |
| `&` | Separator between parameters |


---
## Part 2 — The scaffold

Watch first. You will modify this in Part 3.


In [ ]:
import requests
import pandas as pd

In [ ]:
# 1. Define the endpoint and parameters separately.
#    requests.get() will build the full URL for you.
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 48.14,       # Munich
    "longitude": 11.58,
    "hourly": "temperature_2m"
}

In [ ]:
# 2. Send the GET request.
response = requests.get(url, params=params, timeout=10)

# 3. Always check for errors before touching the response body.
#    If the server returned 4xx or 5xx, this raises an exception immediately.
#    Without this line, a failed request silently returns a response object
#    that looks fine until you try to parse it.
response.raise_for_status()

print("Status code:", response.status_code)
print("Final URL:  ", response.url)

In [ ]:
# 4. Parse the JSON body into a Python dictionary.
data = response.json()

# Inspect the top-level structure.
print("Top-level keys:", list(data.keys()))
print("'hourly' keys: ", list(data["hourly"].keys()))
print("Number of rows:", len(data["hourly"]["time"]))

In [ ]:
# 5. Load into a DataFrame — the hourly dict maps directly.
df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])

print(df.shape)
df.head()

---
## Part 3 — Exercises

Work through both exercises in order. For each one, copy the scaffold from Part 2 and modify only what is necessary.


### Exercise 1 — Add a variable and extend the time window

Fetch **temperature and wind speed** for Munich for the **last 7 days** (plus the default 7-day forecast).

**Hints**
- Open-Meteo accepts multiple variables as a comma-separated string:  
  `"hourly": "temperature_2m,wind_speed_10m"`
- To include past days, add one parameter:  
  `"past_days": 7`
- Expected number of rows: 14 days × 24 hours = **336**

After fetching, confirm the row count and print the first 5 rows.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Data quality check: are the timestamps contiguous?

A common real-world problem is missing rows — gaps in the time series that are invisible unless you check explicitly.

Use `diff()` on the `time` column to compute the interval between consecutive timestamps. If all intervals are equal to 1 hour, the series is contiguous.

**Expected output:** a single unique difference of `0 days 01:00:00`

If you see more than one unique value, there are gaps.


In [ ]:
gaps = df["time"].diff().dropna().unique()
print("Unique time intervals:", gaps)

# Interpretation:
# One value of 1 hour  → no gaps, series is contiguous
# Multiple values      → gaps present, investigate before modelling


---
## Part 4 — What changes across APIs?

The scaffold you used here works for any REST API. The only things that vary are:

| Element | This example | Other APIs |
|---|---|---|
| Endpoint URL | `api.open-meteo.com/v1/forecast` | Different domain and path |
| Parameters | `latitude`, `longitude`, `hourly` | Defined in the API documentation |
| Authentication | None (open API) | API key in `params` or in a request **header** |
| Response structure | `data["hourly"]` | Read the docs to find the right key |

**Authentication example** (for reference — not needed here):
```python
headers = {"Authorization": "Bearer YOUR_API_KEY"}
response = requests.get(url, params=params, headers=headers, timeout=10)
```

**The transferable principle:** read the documentation to find the endpoint, the parameter names, and where to put the key. The four lines — `get`, `raise_for_status`, `json()`, `DataFrame` — stay the same.


---
## Summary

| Concept | One-line takeaway |
|---|---|
| Endpoint | The URL path that identifies a specific resource |
| Query parameters | Key-value pairs that specify what you want |
| `raise_for_status()` | Always call this — fail loudly, not silently |
| JSON → DataFrame | `pd.DataFrame(data["key"])` when the value is a dict of equal-length lists |
| Contiguity check | `df["time"].diff().dropna().unique()` — one value means no gaps |
